In [1]:
import os
from nwtrace import NWTrace
import pandas as pd
import geopandas as gpd
from nwtrace.utils import verify_network_geometry

network_path = "data/more/full_sewers.geojson"
project_crs = "EPSG:26717"

multiple = True
upstream_only = True
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

In [2]:

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls
# target_endpoints = ["OF3729806115"]

outputname_extra = "BC_inlets"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

In [3]:

sewershed = NWTrace(
    network=network_path,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
    crs=project_crs
)


Loading network file: `data\more\full_sewers.geojson`...


In [5]:
fittings = gpd.read_file("data/more/fitting_connections.geojson")
catchbasin_leads = gpd.read_file("data/more/catchbasin_leads.gpkg")



In [ ]:
from nwtrace.utils import *

def repair_node_table(
    nodes: gpd.GeoDataFrame,
    segments: gpd.GeoDataFrame,
    node_field: str = "node_id",
    segment_field: str = "segment_id",
    geometry_field: str = "geometry",
    distance_threshold: float = 0.1
) -> gpd.GeoDataFrame:
    """_summary_

    Parameters
    ----------
    nodes : gpd.GeoDataFrame
        _description_
    segments : gpd.GeoDataFrame
        _description_
    node_field : str, optional
        _description_, by default "node_id"
    segment_field : str, optional
        _description_, by default "segment_id"
    geometry_field : str, optional
        _description_, by default "geometry"
    distance_threshold : float, optional
        _description_, by default 0.1

    Returns
    -------
    gpd.GeoDataFrame
        _description_
    """
    pass

In [4]:

# # additional connections
nodes_up = (fittings[["FACILITYID", "TO_FIXED", "geometry"]]
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .set_index('node_id', drop=False).to_dict(orient="index"))

sewershed.add_upstream_nodes(nodes_up, search_geometry=True)


new_segs = (catchbasin_leads[["FACILITYID", "UP_ASSET_ID", "DN_ASSET_ID"]]
            .rename(columns={"FACILITYID": 'segment_id', "UP_ASSET_ID": 'from', "DN_ASSET_ID": 'to'})
            .set_index('segment_id').to_dict(orient="index"))

sewershed.add_segments(new_segs)

126.04s - invalid syntax (<string>, line 1)
Traceback (most recent call last):
  File "c:\Users\garrett.holmes\AppData\Local\miniconda3\envs\nwtrace\Lib\site-packages\debugpy\_vendored\pydevd\_pydevd_bundle\pydevd_vars.py", line 636, in change_attr_expression
    value = eval(expression, frame.f_globals, frame.f_locals)
  File "<string>", line 1
    node_id segment_id                            geometryCN6363              CN6363  SL1417703   POINT Z (635140.07 4837420.737 0)CN8729              CN8729  SL1421048  POINT Z (633298.151 4837105.562 0)CN4045              CN4045  SL2211821  POINT Z (615099.295 4835640.104 0)CN9525              CN9525     SL4252  POINT Z (644252.554 4851926.449 0)CN4855              CN4855  SL2308097  POINT Z (616092.266 4844100.729 0)...                    ...        ...                                 ...CN6940              CN6940  SL1455797  POINT Z (627066.132 4833215.463 0)CN8814              CN8814  SL1420389  POINT Z (635396.711 4836959.858 0)JP40299031

Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).

Added 120034 node-segment connection(s)
Created 120634 new node(s)
Created 120788 new segment(s).



In [ ]:
if multiple == False:
    result = sewershed.trace_sewershed(
        target_endpoints[0], 
        upstream_only=upstream_only, 
        downstream_only=downstream_only
    )
else:
    result = sewershed.trace_sewersheds(
        target_endpoints, 
        upstream_only=upstream_only, 
        downstream_only=downstream_only, 
    )

Tracing Sewer Network from endpoint(s) [OF3783406347(...)]
	Direction(s): upstream
Preparing directional node connection tree...
Searching Network:


100%|██████████| 190/190 [00:00<00:00, 3638.28it/s]


Found 16 connections overall to all 190 endpoints
Finished!


In [ ]:
d_node, d_seg = sewershed.get_directional_lookup_tables()

In [ ]:
d_seg['CL9478']

{'to': ['MH3836707104'], 'from': ['CB3838007097']}

In [ ]:
d_node['MH3873708965']

{'in': ['SL51551',
  'SL51542',
  'SL53208',
  'CL51548',
  'CL51549',
  'CL51553',
  'CL51554',
  'CL51556',
  'CL51557',
  'CL51399',
  'CL51547',
  'CL491'],
 'out': ['SL53230']}